# Part 4 — GEO / AI Visibility

**Endpoints:**
- `GET /v1/ai-search/overview` — LLM visibility time series (5 engines)
- `GET /v1/domain/aio/keywords-by-target` — AIO keyword presence (snapshot)

**Outputs:**
- `data/processed/ai_visibility_timeseries.csv`
- `data/processed/ai_visibility_snapshot.csv`
- `data/processed/aio_keywords.csv`

In [1]:
import sys, json, time
from pathlib import Path
from datetime import date
import requests
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from config import (
    API_BASE, HEADERS, RATE_LIMIT_DELAY,
    ALL_DOMAINS, DOMAIN_META, SOURCE,
    AI_ENGINES, DATA_RAW, DATA_PROCESSED
)

SNAPSHOT_DATE = str(date.today())

def api_get(path, params=None):
    url = f"{API_BASE}{path}"
    resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
    time.sleep(RATE_LIMIT_DELAY)
    return resp

In [2]:
# ── Subscription / credit check ────────────────────────────────────────────────
sub = api_get("/v1/account/subscription").json().get("subscription_info", {})
status  = sub.get("status", "unknown")
expires = sub.get("expiraton_date", "unknown")
units   = sub.get("units_left", 0)

print(f"Subscription status : {status}")
print(f"Expiration date     : {expires}")
print(f"Units left          : {units}")

if status != "active":
    print(f"\n⚠️  Subscription is '{status}' — data endpoints will return 402.")
    print("   Renew at https://seranking.com/api then re-run this notebook.")

Subscription status : active
Expiration date     : 2026-11-10 19:25:58
Units left          : 99909049.0


## 1. AI Search Overview — all domains × all engines

In [3]:
raw_ai = {}   # domain → engine → response dict

for domain in ALL_DOMAINS:
    meta = DOMAIN_META[domain]
    raw_ai[domain] = {}
    print(f"\n{domain} ({meta['category']}, {meta['size_tier']})")

    for engine in AI_ENGINES:
        resp = api_get("/v1/ai-search/overview", params={
            "target": domain, "scope": "domain",
            "source": SOURCE, "engine": engine
        })
        if resp.status_code == 200:
            data = resp.json()
            raw_ai[domain][engine] = data
            ts   = data.get("time_series", {})
            lp   = ts.get("link_presence", [])
            n    = len(lp)
            earliest = lp[0]["date"]  if lp else "N/A"
            latest   = lp[-1]["date"] if lp else "N/A"
            print(f"  {engine:15s}: {n} months ({earliest} → {latest})")
        else:
            raw_ai[domain][engine] = None
            print(f"  {engine:15s}: {resp.status_code} — {resp.text[:100]}")

(DATA_RAW / "ai_search_raw.json").write_text(json.dumps(raw_ai, indent=2))
print(f"\nRaw saved: {DATA_RAW / 'ai_search_raw.json'}")


coca-cola.com (beverages, large)


  ai-overview    : 19 months (2024-08 → 2026-02)


  chatgpt        : 0 months (N/A → N/A)


  perplexity     : 0 months (N/A → N/A)


  gemini         : 0 months (N/A → N/A)


...

  ai-overview    : 19 months (2024-08 → 2026-02)


  chatgpt        : 0 months (N/A → N/A)


  perplexity     : 0 months (N/A → N/A)


  gemini         : 0 months (N/A → N/A)


  ai-mode        : 0 months (N/A → N/A)

Raw saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/raw/ai_search_raw.json


## 2. Build time-series panel

In [4]:
ts_rows = []

for domain, engine_data in raw_ai.items():
    meta = DOMAIN_META[domain]
    for engine, data in engine_data.items():
        if not data:
            continue
        ts = data.get("time_series", {})

        # Collect all dates present across any time-series key
        all_dates = sorted({
            v["date"]
            for vals in ts.values() if isinstance(vals, list)
            for v in vals if "date" in v
        })

        for d in all_dates:
            row = {
                "domain":    domain,
                "category":  meta["category"],
                "size_tier": meta["size_tier"],
                "engine":    engine,
                "date":      d,
            }
            for key, vals in ts.items():
                if isinstance(vals, list):
                    val_map = {v["date"]: v.get("value") for v in vals if "date" in v}
                    row[key] = val_map.get(d)
            ts_rows.append(row)

df_ai_ts = pd.DataFrame(ts_rows)

if not df_ai_ts.empty:
    df_ai_ts["date"] = pd.to_datetime(df_ai_ts["date"] + "-01", errors="coerce")
    df_ai_ts = df_ai_ts.sort_values(["domain","engine","date"]).reset_index(drop=True)
    print(f"AI time-series: {df_ai_ts.shape}")
    print("Columns:", list(df_ai_ts.columns))
    display(df_ai_ts.head(10))
else:
    print("⚠️  No AI time-series data — subscription may be expired.")

AI time-series: (151, 10)
Columns: ['domain', 'category', 'size_tier', 'engine', 'date', 'link_presence', 'overall_traffic', 'organic_traffic', 'ai_traffic', 'average_position']


,domain,category,size_tier,engine,date,link_presence,overall_traffic,organic_traffic,ai_traffic,average_position
0,coca-cola.com,beverages,large,ai-overview,2024-08-01,0,940800,940800,0,0.00
1,coca-cola.com,beverages,large,ai-overview,2024-09-01,1152,1075065,1075065,0,4.20
2,coca-cola.com,beverages,large,ai-overview,2024-10-01,3249,892510,892510,0,4.77
3,coca-cola.com,beverages,large,ai-overview,2024-11-01,5110,1012551,1012551,0,2.97
4,coca-cola.com,beverages,large,ai-overview,2024-12-01,6850,955742,955742,0,3.00
5,coca-cola.com,beverages,large,ai-overview,2025-01-01,7580,519775,496657,23118,3.11
6,coca-cola.com,beverages,large,ai-overview,2025-02-01,7436,469988,445188,24800,3.25
7,coca-cola.com,beverages,large,ai-overview,2025-03-01,9750,376692,355035,21657,3.56
8,coca-cola.com,beverages,large,ai-overview,2025-04-01,11801,400833,379482,21351,3.95
9,coca-cola.com,beverages,large,ai-overview,2025-05-01,11937,362675,341892,20783,5.44


## 3. Build current-period snapshot

In [5]:
snap_rows = []

for domain, engine_data in raw_ai.items():
    meta = DOMAIN_META[domain]
    for engine, data in engine_data.items():
        if not data:
            continue
        summary = data.get("summary", {})
        row = {
            "domain": domain, "category": meta["category"],
            "size_tier": meta["size_tier"], "engine": engine,
            "snapshot_date": SNAPSHOT_DATE,
        }
        for metric, vals in summary.items():
            if isinstance(vals, dict):
                row[f"{metric}_current"]    = vals.get("current")
                row[f"{metric}_previous"]   = vals.get("previous")
                row[f"{metric}_change_pct"] = vals.get("change_percent")
            else:
                row[metric] = vals
        snap_rows.append(row)

df_ai_snap = pd.DataFrame(snap_rows)
print(f"Snapshot rows: {len(df_ai_snap)}")
if not df_ai_snap.empty:
    display(df_ai_snap.head(10))

Snapshot rows: 40


,domain,category,size_tier,engine,snapshot_date,brand_presence_current,brand_presence_previous,brand_presence_change_pct,link_presence_current,link_presence_previous,link_presence_change_pct,ai_opportunity_traffic_current,ai_opportunity_traffic_previous,ai_opportunity_traffic_change_pct,average_position_current,average_position_previous,average_position_change_pct
0,coca-cola.com,beverages,large,ai-overview,2026-02-27,21.0,None,100,16299,None,100,31576,None,100,4.38,None,None
1,coca-cola.com,beverages,large,chatgpt,2026-02-27,0.0,None,100,0,None,100,0,None,100,NaN,None,None
2,coca-cola.com,beverages,large,perplexity,2026-02-27,0.0,None,100,8,None,100,0,None,100,5.63,None,None
3,coca-cola.com,beverages,large,gemini,2026-02-27,0.0,None,100,18,None,100,0,None,100,2.89,None,None
4,coca-cola.com,beverages,large,ai-mode,2026-02-27,8.0,None,100,2072,None,100,0,None,100,4.33,None,None
5,pepsi.com,beverages,large,ai-overview,2026-02-27,551.0,None,100,30,None,100,29,None,100,3.90,None,None
6,pepsi.com,beverages,large,chatgpt,2026-02-27,NaN,None,100,0,None,100,0,None,100,NaN,None,None
7,pepsi.com,beverages,large,perplexity,2026-02-27,0.0,None,100,0,None,100,0,None,100,NaN,None,None
8,pepsi.com,beverages,large,gemini,2026-02-27,NaN,None,100,0,None,100,0,None,100,NaN,None,None
9,pepsi.com,beverages,large,ai-mode,2026-02-27,127.0,None,100,7,None,100,0,None,100,2.86,None,None


## 4. AIO keyword presence — all domains

In [6]:
aio_rows = []

for domain in ALL_DOMAINS:
    meta = DOMAIN_META[domain]
    resp = api_get("/v1/domain/aio/keywords-by-target", params={
        "target": domain, "scope": "domain", "source": SOURCE, "limit": 100
    })
    if resp.status_code == 200:
        data    = resp.json()
        kw_list = data if isinstance(data, list) else data.get("keywords", [])
        print(f"{domain}: {len(kw_list)} AIO keywords")
        for kw_obj in kw_list:
            kw_text = kw_obj if isinstance(kw_obj, str) else kw_obj.get("keyword", "")
            links   = [] if isinstance(kw_obj, str) else kw_obj.get("links", [])
            aio_rows.append({
                "domain": domain, "category": meta["category"], "size_tier": meta["size_tier"],
                "keyword": kw_text, "featured_urls": json.dumps(links),
                "url_count": len(links), "snapshot_date": SNAPSHOT_DATE
            })
    else:
        print(f"{domain}: {resp.status_code} — {resp.text[:100]}")

df_aio = pd.DataFrame(aio_rows)
print(f"\nTotal AIO rows: {len(df_aio)}")
if not df_aio.empty:
    display(df_aio.head(10))

coca-cola.com: 0 AIO keywords


pepsi.com: 0 AIO keywords


drpepper.com: 0 AIO keywords


drinkolipop.com: 100 AIO keywords


mcdonalds.com: 0 AIO keywords


wendys.com: 0 AIO keywords


shakeshack.com: 100 AIO keywords


fiveguys.com: 0 AIO keywords

Total AIO rows: 200


,domain,category,size_tier,keyword,featured_urls,url_count,snapshot_date
0,drinkolipop.com,beverages,small,prebiotic soda,[],0,2026-02-27
1,drinkolipop.com,beverages,small,probiotic soda,[],0,2026-02-27
2,drinkolipop.com,beverages,small,olipop flavors,[],0,2026-02-27
3,drinkolipop.com,beverages,small,does olipop have caffeine,[],0,2026-02-27
4,drinkolipop.com,beverages,small,is olipop good for you,[],0,2026-02-27
5,drinkolipop.com,beverages,small,olipop ingredients,[],0,2026-02-27
6,drinkolipop.com,beverages,small,drinks without caffeine,[],0,2026-02-27
7,drinkolipop.com,beverages,small,tropical punch,[],0,2026-02-27
8,drinkolipop.com,beverages,small,caffeine free drinks,[],0,2026-02-27
9,drinkolipop.com,beverages,small,non caffeinated drinks,[],0,2026-02-27


## 5. Save all outputs

In [7]:
saved = []

if not df_ai_ts.empty:
    p = DATA_PROCESSED / "ai_visibility_timeseries.csv"
    df_ai_ts.to_csv(p, index=False)
    saved.append(str(p))

if not df_ai_snap.empty:
    p = DATA_PROCESSED / "ai_visibility_snapshot.csv"
    df_ai_snap.to_csv(p, index=False)
    saved.append(str(p))

if not df_aio.empty:
    p = DATA_PROCESSED / "aio_keywords.csv"
    df_aio.to_csv(p, index=False)
    saved.append(str(p))

for s in saved:
    print(f"Saved: {s}")

if not saved:
    print("⚠️  Nothing saved — renew SE Ranking Data API subscription and re-run.")

Saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/processed/ai_visibility_timeseries.csv
Saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/processed/ai_visibility_snapshot.csv
Saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/processed/aio_keywords.csv


## 6. Summary tables

In [8]:
# Time-series depth per engine
if not df_ai_ts.empty:
    depth = df_ai_ts.groupby("engine")["date"].agg(earliest="min", latest="max", months="count").reset_index()
    depth["earliest"] = depth["earliest"].dt.strftime("%Y-%m")
    depth["latest"]   = depth["latest"].dt.strftime("%Y-%m")
    print("=== AI Engine Time-Series Depth ===")
    display(depth)

# Current link_presence pivot: domain × engine
if not df_ai_snap.empty and "link_presence_current" in df_ai_snap.columns:
    pivot = df_ai_snap.pivot_table(
        index=["category","size_tier","domain"],
        columns="engine", values="link_presence_current", aggfunc="first"
    ).reset_index()
    print("\n=== Current AI Link Presence (domain × engine) ===")
    display(pivot)

# AIO keyword count per domain
if not df_aio.empty:
    aio_sum = df_aio.groupby(["category","size_tier","domain"])["keyword"].count().reset_index()
    aio_sum.columns = ["category","size_tier","domain","aio_keyword_count"]
    print("\n=== AIO Keyword Count per Domain (snapshot) ===")
    display(aio_sum.sort_values("aio_keyword_count", ascending=False))
    print(f"\n⚠️  AIO data is cross-sectional (snapshot: {SNAPSHOT_DATE}).")
    print("   AI Search overview time-series may extend back further — check 'earliest' column above.")

=== AI Engine Time-Series Depth ===


,engine,earliest,latest,months
0,ai-overview,2024-08,2026-02,151



=== Current AI Link Presence (domain × engine) ===


engine,category,size_tier,domain,ai-mode,ai-overview,chatgpt,gemini,perplexity
0,beverages,large,coca-cola.com,2072,16299,0,18,8
1,beverages,large,drpepper.com,146,499,0,2,2
2,beverages,large,pepsi.com,7,30,0,0,0
3,beverages,small,drinkolipop.com,116,478,0,2,0
4,fast_food,large,mcdonalds.com,15786,65907,2,76,37
5,fast_food,large,wendys.com,1705,5480,0,1,3
6,fast_food,small,fiveguys.com,883,1186,0,0,0
7,fast_food,small,shakeshack.com,525,898,0,0,0



=== AIO Keyword Count per Domain (snapshot) ===


,category,size_tier,domain,aio_keyword_count
0,beverages,small,drinkolipop.com,100
1,fast_food,small,shakeshack.com,100



⚠️  AIO data is cross-sectional (snapshot: 2026-02-27).
   AI Search overview time-series may extend back further — check 'earliest' column above.
